# Redes neuronales para este dataset

Este notebook sirve para comparar arquitecturas, además de medirlas. La etiqueta es `new`/`used`; la métrica objetivo del concurso es accuracy.

## Cómo calcula una MLP

Cada neurona combina entradas con pesos y un sesgo: $z = W x + b$. ReLU introduce no linealidad en las capas ocultas; la salida softmax convierte los dos puntajes en probabilidades. La entropía cruzada penaliza predicciones equivocadas. Adam actualiza los pesos con mini-batches y early stopping frena cuando deja de mejorar la validación.

```mermaid
flowchart LR
 A["Variables del aviso"] --> B["Escalado + one-hot"]
 B --> C["Capa oculta: ReLU"]
 C --> D["Capa oculta: ReLU"]
 D --> E["Softmax: new / used"]
```

## Modelos, ventajas y desventajas

| Variante | Estructura | Ventaja | Desventaja |
|---|---|---|---|
| MLP superficial | 64 unidades | Pocas capas; baseline legible | Puede quedarse corta en interacciones |
| MLP profunda | 96 → 32 | Aprende varias composiciones no lineales | Más sensible al escalado y regularización |
| MLP regularizada | 128 → 64, L2 mayor | Reduce pesos grandes y overfit | Regularizar demasiado causa underfit |
| MLP texto+SVD | TF-IDF → SVD → MLP | Integra título sin una matriz gigante | La compresión pierde palabras raras |

Ventaja general: las capas densas aprenden combinaciones suaves entre variables. Desventaja para este caso: 70.000 filas, muchas categóricas y reglas tipo “si categoría es X y precio está en este rango”; boosting suele aprender esos cortes con más eficiencia. La evidencia empírica en datos tabulares medianos también advierte que redes estándar pueden quedar detrás de árboles.

## Scores del ciclo de diez intentos

El cuadro y gráfico de abajo usan la misma partición por familias de producto (21.000 filas de validación); el seller encoding se calculó fuera de fold.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
for p in (ROOT,*ROOT.parents):
    if (p/'src'/'ap2026q2_common.py').exists(): ROOT=p; break
sys.path.insert(0,str(ROOT/'src'))
from ap2026q2_common import *
import matplotlib.pyplot as plt
scores = pd.read_csv(ROOT/'results'/'neural_architecture_scores.csv')
display(scores[['model','accuracy','balanced_accuracy','macro_f1','notes'] if 'notes' in scores else ['model','accuracy','balanced_accuracy','macro_f1']])
ax=scores.plot.bar(x='model',y='accuracy',rot=20,ylim=(.85,.96),color=['#3b82f6','#14b8a6','#8b5cf6'])
ax.axhline(.94,color='#ef4444',linestyle='--',label='Meta 0.94')
ax.set_ylabel('Accuracy'); ax.set_title('MLP: accuracy en holdout por familia'); ax.legend(); plt.tight_layout()

In [ ]:
# Diagrama de capas: cada columna representa las neuronas de una capa.
import matplotlib.pyplot as plt
architectures={'Superficial':[5,64,2],'Dos capas':[5,96,32,2],'Profunda':[5,128,64,32,2]}
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,(name,layers) in zip(axes,architectures.items()):
    ax.axis('off'); positions=[]
    for layer,nodes in enumerate(layers):
        draw=min(nodes,7); ys=np.linspace(.12,.88,draw); xs=np.full(draw,layer)
        color='#94a3b8' if layer==0 else ('#f59e0b' if layer==len(layers)-1 else '#3b82f6')
        ax.scatter(xs,ys,s=160,c=color,zorder=3)
        positions.append(list(zip(xs,ys)))
    for j in range(len(positions)-1):
        for x0,y0 in positions[j]:
            for x1,y1 in positions[j+1]: ax.plot([x0,x1],[y0,y1],color='#cbd5e1',lw=.5,zorder=1)
    ax.set_xlim(-.3,len(layers)-.7);ax.set_ylim(0,1);ax.set_title(name+' · '+' → '.join(map(str,layers)))
    ax.set_xticks(range(len(layers)),['input']+['hidden']*(len(layers)-2)+['salida'])
fig.suptitle('Más capas = más transformaciones aprendidas; también más parámetros');plt.tight_layout()

## Entrenar y mirar la curva de aprendizaje

Las configuraciones son intencionalmente pequeñas para que se puedan ejecutar en CPU. La curva de pérdida muestra cómo cae el error de entrenamiento; si sigue bajando mientras la métrica de validación se frena, aparece overfitting. `alpha` es regularización L2; aumentar `alpha` penaliza pesos grandes.

In [ ]:
x, y, _, _ = load_feature_data()
train_idx, valid_idx = holdout_indices(y, test_size=.20, seed=42)
configs=[('Superficial',(64,),.01),('Dos capas',(96,32),.05),('Regularizada',(128,64),.10)]
fit_models=[]; rows=[]
from threadpoolctl import threadpool_limits
for name,layers,alpha in configs:
    with threadpool_limits(limits=1):
        result=evaluate_holdout('mlp_tabular',{'hidden_layer_sizes':layers,'alpha':alpha,'max_iter':35,'batch_size':512,'early_stopping':True,'n_iter_no_change':5})
    fitted=result['model']; model=fitted[1]
    rows.append({'modelo':name,**result['metrics'],'epocas':model.n_iter_})
    fit_models.append((name,model))
display(pd.DataFrame(rows).sort_values('accuracy',ascending=False))
plt.figure(figsize=(8,4))
for name,model in fit_models: plt.plot(model.loss_curve_,label=name)
plt.xlabel('Época');plt.ylabel('Pérdida de entrenamiento');plt.title('Curvas de aprendizaje de las MLP');plt.legend();plt.tight_layout()

## Submission de MLP

La celda siguiente ajusta la variante superficial con todos los registros. Este modelo usa la matriz tabular base del pipeline común; el score del cuadro superior identifica por separado las variantes experimentales con codificación OOF de vendedor.

In [ ]:
GENERATE_SUBMISSION=False
if GENERATE_SUBMISSION:
    output=fit_full_and_submit('mlp_tabular',{'hidden_layer_sizes':(64,),'alpha':.01,'max_iter':35,'batch_size':512,'early_stopping':True,'n_iter_no_change':5})
    print(output)

## Lecturas

- Material de clase: `Intro a Redes para datos estructurados.pdf`, secciones de neurona, backpropagation, loss, mini-batch, Adam, dropout y TabNet.
- Grinsztajn et al., [Why do tree-based models still outperform deep learning on tabular data?](https://arxiv.org/abs/2207.08815).
- Arik & Pfister, [TabNet: Attentive Interpretable Tabular Learning](https://ojs.aaai.org/index.php/AAAI/article/view/16826).